<h1>Ejercicio de Preprocesamiento de datos</h1>

In [ ]:
import pandas as pd
pd.options.display.max_columns = 111
df = pd.read_csv('brooklyn_sales_map.csv',low_memory=False)
df.head(10)

In [ ]:
df.shape

In [ ]:
df.isna().sum()/len(df)*100

In [ ]:
df.dropna(thresh=0.75*len(df),axis=1,inplace=True)

In [ ]:
df.dropna(subset=['building_class_category'],axis=0,inplace=True)

In [ ]:
df.describe()

In [ ]:
df.drop(columns=['Unnamed: 0', 'borough', 'Borough', 'AreaSource', 'block', 'LotArea', 'XCoord', 'YCoord', 'BldgArea', 'BldgClass', 'Easements', 'sale_date','ZipCode', 'CT2010', 'CB2010','Tract2010', 'Sanborn', 'HealthCent', 'ZoneMap', 'SchoolDist', 'TaxMap', 'YearAlter1', 'YearAlter2', 'CondoNo', 'APPBBL', 'PLUTOMapID', 'MAPPLUTO_F', 'Version','land_sqft','Address','UnitsRes','UnitsTotal', 'ComArea', 'ResArea', 'OfficeArea','RetailArea','GarageArea', 'StrgeArea', 'FactryArea', 'OtherArea'], axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
cat_cols = df.select_dtypes('object').columns
k = 0
cardinality_cols = []
for col in cat_cols:
    value_counts = df[col].value_counts(normalize=True).round(2)
    if len(value_counts) > 5 or len(value_counts) < 2:
        cardinality_cols.append(col)#agrega elementos a la lista
    else:
        print("======================================================")
        print(col)
        print(value_counts)
cardinality_cols.remove('neighborhood')
cardinality_cols.remove('building_class_at_sale')
cardinality_cols.remove('tax_class')
print(f"Removed Columns: {cardinality_cols}")
df.drop(columns=cardinality_cols, axis=1, inplace=True)

In [ ]:
df['neighborhood'].value_counts(normalize=True).round(2)

In [ ]:
import numpy as np
neighborhood = df ['neighborhood'].value_counts(normalize=True).round(2)
idx = np.where(neighborhood > 0.01)[0].max()
places = neighborhood.index[:idx]
df ['neighborhood'] =df ['neighborhood'].apply(lambda x: x if x in places else 'OTRONEIG')
df ['neighborhood'].value_counts(normalize=True)

In [ ]:
df['building_class_at_sale'].value_counts(normalize=True).round(2)

In [ ]:
neighborhood = df ['building_class_at_sale'].value_counts(normalize=True).round(2)
idx = np.where(neighborhood > 0.01)[0].max()
places = neighborhood.index[:idx]
df ['building_class_at_sale'] =df ['building_class_at_sale'].apply(lambda x: x if x in places else 'OTRACLASS')
df ['building_class_at_sale'].value_counts(normalize=True)

In [ ]:
num_cols = df.select_dtypes('number').columns

low_var_cols = []
for col in num_cols:
    scaled = (df[col] - df[col].mean()) / df[col].std()
    variance = scaled.var().round(4)
    if variance == 0 or df[col].std() == 0:
        low_var_cols.append(col)
#     else:
#         print(col, variance)

df.drop(columns=low_var_cols, axis=1, inplace=True)
print(f"Removed Columns: {low_var_cols}")
print(df.shape, len(low_var_cols))

In [ ]:
import matplotlib.pyplot as plt
plt.hist(df["sale_price"],edgecolor="black",linewidth=1)
plt.show()

In [ ]:
plt.boxplot(df["sale_price"])

In [ ]:
mask = (df['sale_price'] > 100000)
df = df [mask]

In [ ]:
df.shape

In [ ]:
Q1 = df["sale_price"].quantile(0.25)
Q3 = df["sale_price"].quantile(0.75)
IQR = Q3 - Q1
print("Primer cuartil:", Q1)
print("Tercer cuartil:", Q3)
print("Rango intercuartil (IQR):", IQR)

In [ ]:
BI=Q1-1.5*IQR
BS=Q3+1.5*IQR
print("Bigote inferior:",BI)
print("Bigote superior:",BS)

In [ ]:
df.drop(df.loc[(df["sale_price"]<BI) | (df["sale_price"]>BS)].index, inplace=True)

In [ ]:
df.shape

In [ ]:
plt.hist(df["sale_price"],edgecolor="black",linewidth=1)
plt.show()

In [ ]:
plt.boxplot(df["sale_price"])

In [ ]:
df['age'] = 2023 - (df['year_built'] + df['YearBuilt']) // 2
df.drop(['zip_code', 'year_built', 'YearBuilt', 'year_of_sale'], axis=1, inplace=True)

In [ ]:
num_columns=['ProxCode','IrrLotCode','building_class_at_sale']
for i in num_columns:
    df[i]=df[i].fillna(df[i].mode()[0])

In [ ]:
print(df ['tax_class'].unique())
print(df ['tax_class'].isnull().sum())
print(df ['tax_class_at_sale'].unique())
print(df ['tax_class_at_sale'].isnull().sum())

In [ ]:
df2 = df.groupby(['building_class_at_sale'])['building_class_at_sale'].count()
print(df2)

In [ ]:
df['tax_class'].fillna(df ['tax_class_at_sale'], inplace=True)
del df ['tax_class_at_sale']

In [ ]:
df ['tax_class'] = df ['tax_class'].map({'1A':'1A','1B':'1B','1C':'1C','2A':'2A','2B':'2B','2C':'2C',
                                         '4':'4','3':'3','2':'2','1':'1',3:'3',4:'4',2:'2',1:'1'})
print(df ['tax_class'].unique())
print(df ['tax_class'].isnull().sum())

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
print(df ['SplitZone'].unique())
df ['SplitZone']= df ['SplitZone'].fillna(value='N')
df ['SplitZone']= df ['SplitZone'].astype('category')
df ['SplitZone']= df ['SplitZone'].cat.codes
print(df['SplitZone'].unique())

In [ ]:
import numpy as np
numeric = df.select_dtypes(include=np.number)
numeric_columns = numeric.columns
df[numeric_columns] = df[numeric_columns].fillna(df.median(numeric_only= True))

In [ ]:
df.isna().sum()/len(df)*100

In [ ]:
df['IrrLotCode']= df['IrrLotCode'].astype('category')

In [ ]:
cat_columns = df.select_dtypes(['category']).columns
print(cat_columns)
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

In [ ]:
# Matriz de correlación
# ==============================================================================
X=df.drop(['sale_price','neighborhood','tax_class','ProxCode','building_class_at_sale'],axis=1)
corr_matrix = X.corr(method='pearson')
corr_matrix

In [ ]:
def tidy_corr_matrix(corr_mat):
    '''
    Función para convertir una matriz de correlación de pandas en formato tidy.
    '''
    corr_mat = corr_mat.stack().reset_index()
    corr_mat.columns = ['variable_1','variable_2','r']
    corr_mat = corr_mat.loc[corr_mat['variable_1'] != corr_mat['variable_2'], :]
    corr_mat['abs_r'] = np.abs(corr_mat['r'])
    corr_mat = corr_mat.sort_values('abs_r', ascending=False)
    
    return(corr_mat)

tidy_corr_matrix(corr_matrix).head(10)

In [ ]:
import seaborn as sns
# Heatmap matriz de correlaciones
# ==============================================================================
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(22, 22))

sns.heatmap(
    corr_matrix,
    annot     = True,
    cbar      = False,
    annot_kws = {"size": 8},
    vmin      = -1,
    vmax      = 1,
    center    = 0,
    cmap      = sns.diverging_palette(20, 220, n=200),
    square    = True,
    ax        = ax
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation = 45,
    horizontalalignment = 'right',
)

ax.tick_params(labelsize = 10)

In [ ]:
corr = X.corr()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(np.bool_))
drop = [column for column in upper.columns if any(np.abs(upper[column]) > 0.7)]
print(f"columns dropped are: {drop}")
df.drop(columns=drop, axis=1, inplace=True)
X .drop(columns=drop, axis=1, inplace=True)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del df ['SanitBoro']
del X ['SanitBoro']
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del df['CD']
del X['CD']
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del df ['LotType']
del X ['LotType']
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del df ['LotDepth']
del X ['LotDepth']
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del df ['LandUse']
del X ['LandUse']
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
del df ['NumFloors']
del X ['NumFloors']
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
vif.round(1)

In [ ]:
viz_df = df.copy()
median_home_price = viz_df['sale_price'].median().round(2)
print(median_home_price)
k = 0
plt.figure(figsize=(20, 30))
for col in viz_df.drop('sale_price', axis=1).columns:
    plt.subplot(6, 3, k + 1)
    if viz_df[col].dtype == 'float64' or viz_df[col].dtype =='int8':
        sns.scatterplot(data=viz_df, x=col, y='sale_price')
        plt.title(f"Sale Price Vs {col}")
    else:
        group = viz_df[[col, 'sale_price']].groupby(by=col).median()
        sns.barplot(data=group, x=group.index, y='sale_price')
        plt.axhline(y=median_home_price, label='Median Home Price', color='black', linestyle='--', linewidth=0.7)
        plt.title(f"Avg Home Price by {col}")
        plt.legend()
    k += 1
plt.tight_layout()

In [ ]:
data = pd.get_dummies(df.neighborhood)
df = pd.concat([df.drop("neighborhood",axis="columns"), data], axis="columns")
df.head()

In [ ]:
data = pd.get_dummies(df.tax_class)
df = pd.concat([df.drop("tax_class",axis="columns"), data], axis="columns")
df.head()

In [ ]:
data = pd.get_dummies(df.ProxCode)
df = pd.concat([df.drop("ProxCode",axis="columns"), data], axis="columns")
df.head()

In [ ]:
data = pd.get_dummies(df.building_class_at_sale)
df = pd.concat([df.drop("building_class_at_sale",axis="columns"), data], axis="columns")
df.head()

In [ ]:
df.shape

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df.to_numpy())
df_scaled = pd.DataFrame(df_scaled,columns=df.columns) 

In [ ]:
df_scaled.head()

In [ ]:
df_scaled.columns

In [ ]:
#df_scaled.rename(columns={'1.0': "1.0", '2.0': "2.0",'3.0':"3.0"})
df_scaled.columns = df_scaled.columns.astype(str)

In [ ]:
#Reparto de datos en train y test
# ==============================================================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
                                        df_scaled.drop('sale_price', axis = 'columns'),
                                        df_scaled['sale_price'],
                                        train_size   = 0.8,
                                        random_state = 1234,
                                        shuffle      = True
                                    )

In [ ]:
from sklearn.neural_network import MLPRegressor
# Instantiate MLPRegressor
#
nn = MLPRegressor(
    activation='relu',
    hidden_layer_sizes=(30, 200),
    alpha=0.001,
    random_state=20,
    early_stopping=False
)
#
# Train the model
#
nn.fit(X_train, y_train)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error
 
# Make prediction
pred = nn.predict(X_test)
#
# Calculate accuracy and error metrics
#
test_set_rsquared = nn.score(X_test, y_test)
test_set_rmse = np.sqrt(mean_squared_error(y_test, pred))
#
# Print R_squared and RMSE value
#
print('R_squared value: ', test_set_rsquared)
print('RMSE: ', test_set_rmse)